# Hands-On Introduction to Open AI Function Calling

[Article](https://todatabeyond.com/blog/hands-on-introduction-to-open-ai-function-calling). Original examples are preserved in the nonexecuting appendix below.

Requires an `OPENAI_API_KEY` in your environment; calls incur charges. `MODEL` defaults to `gpt-5.4-mini`, the tested model. Do not place credentials in cells.

Install: `pip install openai==2.43.0 jsonschema==4.26.0 python-dotenv==1.2.2`.

In [ ]:
import os, json, sys
from importlib.metadata import version
from openai import OpenAI
from jsonschema import validate, ValidationError
from dotenv import load_dotenv
load_dotenv()
MODEL = os.getenv("MODEL", "gpt-5.4-mini")
client = OpenAI(max_retries=0, timeout=45)
def chat(messages, **kwargs):
    return client.chat.completions.create(
        model=MODEL, messages=messages, max_completion_tokens=1024,
        reasoning_effort="none", **kwargs)
print("Python:", sys.version.split()[0])
print("openai:", version("openai"), "jsonschema:", version("jsonschema"))
print("Model:", MODEL)

In [ ]:
def get_current_weather(location, unit="fahrenheit"):
    if unit not in {"fahrenheit", "celsius"}:
        raise ValueError("Unsupported temperature unit")
    if not isinstance(location, str) or location.lower() not in {"boston", "boston, ma", "boston, massachusetts"}:
        raise ValueError("The fixed teaching fixture supports Boston only")
    temperature = 72 if unit == "fahrenheit" else round((72 - 32) * 5 / 9, 2)
    return {"location": location, "temperature": temperature, "unit": unit,
            "forecast": ["sunny", "windy"]}
print(json.dumps(get_current_weather("Boston, MA"), indent=2))

In [ ]:
weather_schema = {
    "type": "object",
    "properties": {
        "location": {"type": "string", "description": "City and state, for example Boston, MA"},
        "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]}
    },
    "required": ["location", "unit"], "additionalProperties": False
}
tools = [{"type": "function", "function": {
    "name": "get_current_weather", "description": "Get weather for the requested city. Use fahrenheit unless the user asks for celsius.",
    "strict": True, "parameters": weather_schema
}}]
print(json.dumps(tools, indent=2))

In [ ]:
messages = [{"role": "developer", "content": "Demonstrate a weather tool using a fixed teaching fixture. When given a tool result, summarize its temperature, unit and forecast. Call it an example, not live weather."},
            {"role": "user", "content": "What is the weather like in Boston, MA?"}]
print(json.dumps(messages, indent=2))

In [ ]:
response = chat(messages, tools=tools, tool_choice="auto", parallel_tool_calls=False)
print(response.model_dump_json(indent=2))

In [ ]:
response_message = response.choices[0].message
assert len(response_message.tool_calls or []) == 1
tool_call = response_message.tool_calls[0]
assert tool_call.function.name == "get_current_weather"
print(response_message.model_dump_json(indent=2))

In [ ]:
arguments = json.loads(tool_call.function.arguments)
validate(arguments, weather_schema)
print(json.dumps(arguments, indent=2))

In [ ]:
observation = get_current_weather(**arguments)
assert isinstance(observation["location"], str)
print(json.dumps(observation, indent=2))

In [ ]:
greeting = [{"role": "user", "content": "hi!"}]
greeting_response = chat(greeting, tools=tools, parallel_tool_calls=False)
assert not greeting_response.choices[0].message.tool_calls
print(greeting_response.choices[0].message.model_dump_json(indent=2))

In [ ]:
auto_response = chat(greeting, tools=tools, tool_choice="auto", parallel_tool_calls=False)
assert not auto_response.choices[0].message.tool_calls
print(auto_response.choices[0].message.model_dump_json(indent=2))

In [ ]:
none_response = chat(greeting, tools=tools, tool_choice="none")
assert not none_response.choices[0].message.tool_calls
print(none_response.choices[0].message.model_dump_json(indent=2))

In [ ]:
weather_without_tools = chat(messages, tools=tools, tool_choice="none")
assert not weather_without_tools.choices[0].message.tool_calls
print(weather_without_tools.choices[0].message.model_dump_json(indent=2))

In [ ]:
forced_choice = {"type": "function", "function": {"name": "get_current_weather"}}
forced_response = chat(messages, tools=tools, tool_choice=forced_choice,
                       parallel_tool_calls=False)
assert len(forced_response.choices[0].message.tool_calls or []) == 1
print(forced_response.choices[0].message.model_dump_json(indent=2))

In [ ]:
forced_greeting = chat(greeting, tools=tools, tool_choice=forced_choice,
                       parallel_tool_calls=False)
assert len(forced_greeting.choices[0].message.tool_calls or []) == 1
print(forced_greeting.choices[0].message.model_dump_json(indent=2))
print("Not dispatched: the user has not supplied a location.")

In [ ]:
response_message = forced_response.choices[0].message
tool_call = response_message.tool_calls[0]
if tool_call.function.name != "get_current_weather":
    raise ValueError("Unknown function")
arguments = json.loads(tool_call.function.arguments)
validate(arguments, weather_schema)
observation = get_current_weather(**arguments)
print(json.dumps(observation, indent=2))

In [ ]:
conversation = messages + [response_message.model_dump(exclude_none=True),
    {"role": "tool", "tool_call_id": tool_call.id,
     "content": json.dumps(observation)}]
print(json.dumps(conversation, indent=2))

In [ ]:
final_response = chat(conversation, tools=tools, tool_choice="none")
assert final_response.choices[0].message.content
assert "72" in final_response.choices[0].message.content
assert all(word in final_response.choices[0].message.content.lower() for word in ["sunny", "windy"])
print(final_response.choices[0].message.content)

In [ ]:
token_probe = [{"role": "user", "content": "Reply with the word hello."}]
plain = chat(token_probe)
with_tools = chat(token_probe, tools=tools, tool_choice="none")
print(json.dumps({"without_tools_prompt_tokens": plain.usage.prompt_tokens,
                  "with_tools_prompt_tokens": with_tools.usage.prompt_tokens,
                  "difference": with_tools.usage.prompt_tokens - plain.usage.prompt_tokens}, indent=2))

In [ ]:
assert get_current_weather("Boston", "fahrenheit")["temperature"] == 72
assert get_current_weather("Boston", "celsius")["temperature"] == 22.22
for bad in [{"location": {"location": "Boston"}, "unit": "fahrenheit"},
            {"location": "Boston", "unit": "kelvin"}, {"location": "Boston"}]:
    try: validate(bad, weather_schema)
    except ValidationError: pass
    else: raise AssertionError("Bad arguments accepted")
assert conversation[-1]["tool_call_id"] == conversation[-2]["tool_calls"][0]["id"]
assert not final_response.choices[0].message.tool_calls
assert with_tools.usage.prompt_tokens > plain.usage.prompt_tokens
print("PASS: both temperature units, invalid argument rejection, tool-call IDs, final answer and measured schema-token overhead.")

## Original archive appendix — not executed

The original code, prompts and transcripts below are retained verbatim for comparison. Historical SDK calls, placeholders and setup instructions are not the runnable path.

### Original block 1

```text
pip install openai
```

### Original block 2

```text
import os
import openai

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file
openai.api_key = os.environ['OPENAI_API_KEY']
```

### Original block 3

```text
import json

# In production, this could be your backend API or an external API

def get_current_weather(location, unit="fahrenheit"):
    """Get the current weather in a given location"""
    weather_info = {
        "location": location,
        "temperature": "72",
        "unit": unit,
        "forecast": ["sunny", "windy"],
    }
    return json.dumps(weather_info)
```

### Original block 4

```text
# define a function
functions = [
    {
        "name": "get_current_weather",
        "description": "Get the current weather in a given location",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "The city and state, e.g. San Francisco, CA",
                },
                "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
            },
            "required": ["location"],
        },
    }
]
```

### Original block 5

```text
messages = [
    {
        "role": "user",
        "content": "What's the weather like in Boston?"
    }
]
```

### Original block 6

```text
import openai
response = openai.ChatCompletion.create(
    model="gpt-3.5-turbo-0613",
    messages=messages,
    functions=functions
)
print(response)
```

### Original block 7

```text
response_message = response["choices"][0]["message"]
```

### Original block 8

```text
json.loads(response_message["function_call"]["arguments"])
```

### Original block 9

```text
args = json.loads(response_message["function_call"]["arguments"])
get_current_weather(args)
```

### Original block 10

```text
messages = [
    {
        "role": "user",
        "content": "hi!",
    }
]

response = openai.ChatCompletion.create(
    model="gpt-3.5-turbo-0613",
    messages=messages,
    functions=functions,
)

print(response)
```

### Original block 11

```text
messages = [
    {
        "role": "user",
        "content": "hi!",
    }
]
response = openai.ChatCompletion.create(
    model="gpt-3.5-turbo-0613",
    messages=messages,
    functions=functions,
    function_call="auto",
)
print(response)
```

### Original block 12

```text
messages = [
    {
        "role": "user",
        "content": "hi!",
    }
]
response = openai.ChatCompletion.create(
    model="gpt-3.5-turbo-0613",
    messages=messages,
    functions=functions,
    function_call="none",
)
print(response)
```

### Original block 13

```text
messages = [
    {
        "role": "user",
        "content": "What's the weather in Boston?",
    }
]
response = openai.ChatCompletion.create(
    model="gpt-3.5-turbo-0613",
    messages=messages,
    functions=functions,
    function_call="none",
)
print(response)
```

### Original block 14

```text
messages = [
    {
        "role": "user",
        "content": "What's the weather like in Boston!",
    }
]
response = openai.ChatCompletion.create(
    model="gpt-3.5-turbo-0613",
    messages=messages,
    functions=functions,
    function_call={"name": "get_current_weather"},
)
print(response)
```

### Original block 15

```text
messages = [
    {
        "role": "user",
        "content": "hi!",
    }
]
response = openai.ChatCompletion.create(
    model="gpt-3.5-turbo-0613",
    messages=messages,
    functions=functions,
    function_call={"name": "get_current_weather"},
)
print(response)
```

### Original block 16

```text
messages = [
    {
        "role": "user",
        "content": "What's the weather like in Boston!",
    }
]
response = openai.ChatCompletion.create(
    model="gpt-3.5-turbo-0613",
    messages=messages,
    functions=functions,
    function_call={"name": "get_current_weather"},
)

args = json.loads(response["choices"][0]["message"]['function_call']['arguments'])
observation = get_current_weather(args)
```

### Original block 17

```text
messages.append(
        {
            "role": "function",
            "name": "get_current_weather",
            "content": observation,
        }
)
```

### Original block 18

```text
response = openai.ChatCompletion.create(
    model="gpt-3.5-turbo-0613",
    messages=messages,
)
print(response)
```

### Original prompt/transcript 1

```text
{
 “id”: “chatcmpl-9TQ933VAIQwAXGDoK5QROsKq3dsVt”,
 “object”: “chat.completion”,
 “created”: 1716799041,
 “model”: “gpt-3.5-turbo-0613”,
 “choices”: [
 {
 “index”: 0,
 “message”: {
 “role”: “assistant”,
 “content”: null,
 “function_call”: {
 “name”: “get_current_weather”,
 “arguments”: “{\n \”location\”: \”Boston, MA\”\n}”
 }
 },
 “logprobs”: null,
 “finish_reason”: “function_call”
 }
 ],
 “usage”: {
 “prompt_tokens”: 82,
 “completion_tokens”: 18,
 “total_tokens”: 100
 },
 “system_fingerprint”: null
}
```

### Original prompt/transcript 2

```text
<OpenAIObject at 0x7fee64221f90> JSON: {
 “role”: “assistant”,
 “content”: null,
 “function_call”: {
 “name”: “get_current_weather”,
 “arguments”: “{\n \”location\”: \”Boston, MA\”\n}”
 }
}
```

### Original prompt/transcript 3

```text
{‘location’: ‘Boston, MA’}
```

### Original prompt/transcript 4

```text
‘{“location”: {“location”: “Boston, MA”}, “temperature”: “72”, “unit”: “fahrenheit”, “forecast”: [“sunny”, “windy”]}’
```

### Original prompt/transcript 5

```text
{
 “id”: “chatcmpl-9TQ95iiWHENQCVjVKRM7jzPn8G60p”,
 “object”: “chat.completion”,
 “created”: 1716799043,
 “model”: “gpt-3.5-turbo-0613”,
 “choices”: [
 {
 “index”: 0,
 “message”: {
 “role”: “assistant”,
 “content”: “Hello! How can I assist you today?”
 },
 “logprobs”: null,
 “finish_reason”: “stop”
 }
 ],
 “usage”: {
 “prompt_tokens”: 76,
 “completion_tokens”: 10,
 “total_tokens”: 86
 },
 “system_fingerprint”: null
}
```

### Original prompt/transcript 6

```text
{
 “id”: “chatcmpl-9TQ97i7tWoviRuECqrijLyWSEiFtm”,
 “object”: “chat.completion”,
 “created”: 1716799045,
 “model”: “gpt-3.5-turbo-0613”,
 “choices”: [
 {
 “index”: 0,
 “message”: {
 “role”: “assistant”,
 “content”: “Hello! How can I assist you today?”
 },
 “logprobs”: null,
 “finish_reason”: “stop”
 }
 ],
 “usage”: {
 “prompt_tokens”: 77,
 “completion_tokens”: 9,
 “total_tokens”: 86
 },
 “system_fingerprint”: null
}
```

### Original prompt/transcript 7

```text
{
 “id”: “chatcmpl-9TRg6WIMf2ClbP7MM0EA7mtM76Zty”,
 “object”: “chat.completion”,
 “created”: 1716804934,
 “model”: “gpt-3.5-turbo-0613”,
 “choices”: [
 {
 “index”: 0,
 “message”: {
 “role”: “assistant”,
 “content”: “Let me check the current weather in Boston for you.”
 },
 “logprobs”: null,
 “finish_reason”: “stop”
 }
 ],
 “usage”: {
 “prompt_tokens”: 82,
 “completion_tokens”: 11,
 “total_tokens”: 93
 },
 “system_fingerprint”: null
}
```

### Original prompt/transcript 8

```text
messages = [
 {
 “role”: “user”,
 “content”: “What’s the weather like in Boston!”,
 }
]
response = openai.ChatCompletion.create(
 model=”gpt-3.5-turbo-0613",
 messages=messages,
 functions=functions,
 function_call={“name”: “get_current_weather”},
)
print(response)
messages = [
 {
 “role”: “user”,
 “content”: “What’s the weather like in Boston!”,
 }
]
response = openai.ChatCompletion.create(
 model=”gpt-3.5-turbo-0613",
 messages=messages,
 functions=functions,
 function_call={“name”: “get_current_weather”},
)
print(response)
{
 “id”: “chatcmpl-9TRg7lo4DZnOHYfoGQHgiBK6ZSnMn”,
 “object”: “chat.completion”,
 “created”: 1716804935,
 “model”: “gpt-3.5-turbo-0613”,
 “choices”: [
 {
 “index”: 0,
 “message”: {
 “role”: “assistant”,
 “content”: null,
 “function_call”: {
 “name”: “get_current_weather”,
 “arguments”: “{\n\”location\”: \”Boston, MA\”\n}”
 }
 },
 “logprobs”: null,
 “finish_reason”: “stop”
 }
 ],
 “usage”: {
 “prompt_tokens”: 89,
 “completion_tokens”: 10,
 “total_tokens”: 99
 },
 “system_fingerprint”: null
}
```

### Original prompt/transcript 9

```text
{
 “id”: “chatcmpl-9TRg7JqPqIEF0aYBT7qrJDmznarQC”,
 “object”: “chat.completion”,
 “created”: 1716804935,
 “model”: “gpt-3.5-turbo-0613”,
 “choices”: [
 {
 “index”: 0,
 “message”: {
 “role”: “assistant”,
 “content”: null,
 “function_call”: {
 “name”: “get_current_weather”,
 “arguments”: “{\n \”location\”: \”San Francisco, CA\”\n}”
 }
 },
 “logprobs”: null,
 “finish_reason”: “stop”
 }
 ],
 “usage”: {
 “prompt_tokens”: 83,
 “completion_tokens”: 12,
 “total_tokens”: 95
 },
 “system_fingerprint”: null
}
```

### Original prompt/transcript 10

```text
{
 “id”: “chatcmpl-9TUxfK9e08r9GWcg18d8qngKy9kbr”,
 “object”: “chat.completion”,
 “created”: 1716817555,
 “model”: “gpt-3.5-turbo-0613”,
 “choices”: [
 {
 “index”: 0,
 “message”: {
 “role”: “assistant”,
 “content”: “The weather in Boston is currently sunny and windy with a temperature of 72 degrees Fahrenheit.”
 },
 “logprobs”: null,
 “finish_reason”: “stop”
 }
 ],
 “usage”: {
 “prompt_tokens”: 76,
 “completion_tokens”: 18,
 “total_tokens”: 94
 },
 “system_fingerprint”: null
}
```